# HR Turnover Prediction - AI Solution
## TrustedAI x RH Hackathon

**Themes:** Cybersecurity + Explainable AI

---

### Project Overview

This notebook presents an AI solution to help HR identify employees at risk of resignation and explain why.

**Objectives:**
- Predict which employees are likely to leave
- Explain predictions transparently (XAI)
- Ensure data privacy (Cybersecurity / GDPR)
- Audit for algorithmic fairness

## 1. Setup & Dependencies

In [9]:
# Install required packages - numpy<2 must be installed first for compatibility
!pip install "numpy<2" pandas scikit-learn shap matplotlib seaborn -q

In [10]:
import os
import hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Ensure output directory exists
os.makedirs('outputs', exist_ok=True)
print('Setup complete')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## 2. Data Loading & Exploration

In [ ]:
df = pd.read_csv('data/hr_data.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumns:\n{df.columns.tolist()}')

In [ ]:
df.head(3)

In [ ]:
print('Target Distribution (Termd):')
print(df['Termd'].value_counts())
print(f'\nTurnover Rate: {df["Termd"].mean()*100:.1f}%')

# NOTE: The columns below are EXCLUDED from modelling because they
# directly reveal whether someone was terminated (data leakage).
# Including EmpStatusID caused 100% accuracy in the original notebook.
print('\nLeaky columns identified and excluded from modelling:')
print('  EmpStatusID       - encodes Termd directly')
print('  EmploymentStatus  - text version of EmpStatusID')
print('  DateofTermination - exists only for terminated employees')
print('  TermReason        - exists only for terminated employees')

## 3. Data Anonymization (Cybersecurity / GDPR)

In [ ]:
# Step 1: Remove direct identifiers (PII)
PII_COLUMNS = ['Employee_Name', 'EmpID', 'ManagerName', 'DOB', 'Zip']

# Step 2: Remove target-leaking columns
LEAKING_COLUMNS = ['EmpStatusID', 'EmploymentStatus', 'DateofTermination', 'TermReason']

df_clean = df.drop(columns=PII_COLUMNS + LEAKING_COLUMNS, errors='ignore')

# Step 3: Pseudonymise quasi-identifiers with a salted SHA-256 hash.
# Python's built-in hash() is NOT suitable for GDPR pseudonymisation:
# it is deterministic per process, not one-way, and reversible.
# SHA-256 with a secret salt is one-way and non-reversible (GDPR Art. 25).
_SALT = b'hackathon_2025_secret_salt_do_not_share'

def sha256_pseudonymise(series):
    """Replace each value with a truncated (8-char) SHA-256 hex digest."""
    def _hash(val):
        return hashlib.sha256(_SALT + str(val).encode()).hexdigest()[:8]
    return series.apply(_hash)

for col in ['Position', 'Department', 'RecruitmentSource']:
    if col in df_clean.columns:
        df_clean[col + '_Hash'] = sha256_pseudonymise(df_clean[col])

print('PII removed:', PII_COLUMNS)
print('Leaky features removed:', LEAKING_COLUMNS)
print('Quasi-identifiers pseudonymised with salted SHA-256')
print(f'Dataset shape after anonymization: {df_clean.shape}')

NameError: name 'df' is not defined

## 4. Feature Engineering

In [ ]:
# Use a fixed reference date so results are reproducible
REFERENCE_DATE = pd.Timestamp('2020-01-01')
df_feat = df_clean.copy()

# --- Date-based features ---
if 'DateofHire' in df_feat.columns:
    df_feat['DateofHire'] = pd.to_datetime(df_feat['DateofHire'], errors='coerce')
    df_feat['Tenure_Days'] = (
        (REFERENCE_DATE - df_feat['DateofHire']).dt.days.clip(lower=0)
    )

if 'LastPerformanceReview_Date' in df_feat.columns:
    df_feat['LastPerformanceReview_Date'] = pd.to_datetime(
        df_feat['LastPerformanceReview_Date'], errors='coerce')
    df_feat['DaysSinceReview'] = (
        (REFERENCE_DATE - df_feat['LastPerformanceReview_Date']).dt.days.clip(lower=0)
    )

# --- Binary encoding ---
if 'HispanicLatino' in df_feat.columns:
    df_feat['HispanicLatino_Enc'] = (
        df_feat['HispanicLatino'].str.strip()
        .map({'Yes': 1, 'No': 0, 'Y': 1, 'N': 0})
        .fillna(0).astype(int)
    )

# --- Label encoding for low-cardinality categoricals ---
for col in ['CitizenDesc', 'State', 'PerformanceScore', 'MaritalDesc']:
    if col in df_feat.columns:
        df_feat[col + '_Enc'] = LabelEncoder().fit_transform(df_feat[col].astype(str))

print('Feature engineering complete.')
print('New columns: Tenure_Days, DaysSinceReview, HispanicLatino_Enc,')
print('  CitizenDesc_Enc, State_Enc, PerformanceScore_Enc, MaritalDesc_Enc')

NameError: name 'pd' is not defined

In [ ]:
# Assemble feature matrix
# Sensitive demographic columns (Sex, RaceDesc) are preserved only for
# the fairness audit (Section 7) and are NOT fed into the model.
FEATURE_CANDIDATES = [
    'MarriedID', 'MaritalStatusID', 'GenderID',
    'DeptID', 'PerfScoreID', 'PayRate', 'PositionID',
    'FromDiversityJobFairID', 'ManagerID',
    'EngagementSurvey', 'EmpSatisfaction',
    'SpecialProjectsCount', 'DaysLateLast30',
    'Tenure_Days', 'DaysSinceReview',
    'HispanicLatino_Enc',
    'CitizenDesc_Enc', 'State_Enc',
    'PerformanceScore_Enc', 'MaritalDesc_Enc',
]

available_features = [f for f in FEATURE_CANDIDATES if f in df_feat.columns]

X = df_feat[available_features].fillna(0)
y = df_feat['Termd']

print(f'Feature count : {len(available_features)}')
print(f'Features      : {available_features}')
print(f'Dataset shape : {X.shape}')
print(f'\nTarget distribution:\n{y.value_counts()}')

## 5. Model Training & Evaluation

In [ ]:
# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest with class balancing to handle the 2:1 imbalance
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=' * 55)
print('Hold-out Test Set Performance')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['Stay', 'Leave']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.3f}')

In [ ]:
# 5-Fold Stratified Cross-Validation — more reliable on 310 rows
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    model, X, y, cv=cv,
    scoring=['accuracy', 'f1_macro', 'roc_auc'],
    return_train_score=False
)

print('5-Fold Cross-Validation Results:')
for metric in ['test_accuracy', 'test_f1_macro', 'test_roc_auc']:
    scores = cv_results[metric]
    label  = metric.replace('test_', '').replace('_', ' ').upper()
    print(f'  {label:14s}: {scores.mean():.3f} +/- {scores.std():.3f}')

In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Stay', 'Leave'],
    yticklabels=['Stay', 'Leave'],
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Test Set)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name='Random Forest')
axes[1].set_title('ROC Curve (Test Set)')
axes[1].plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random baseline')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/model_evaluation.png', dpi=150)
plt.show()
print('Plots saved to outputs/model_evaluation.png')

## 6. Explainable AI (XAI) — SHAP Analysis

In [11]:
# Global SHAP feature importance
explainer   = shap.TreeExplainer(model)
shap_values = explainer(X_test)   # shap.Explanation object

# Handle both old (list) and new (ndarray with 3 dims) SHAP API
sv = shap_values.values
if sv.ndim == 3:
    sv_class1 = sv[:, :, 1]
    base_vals  = shap_values.base_values[:, 1]
else:
    sv_class1 = sv
    base_vals  = shap_values.base_values

shap_exp_class1 = shap.Explanation(
    values=sv_class1,
    base_values=base_vals,
    data=shap_values.data,
    feature_names=available_features
)

plt.figure(figsize=(10, 6))
shap.summary_plot(sv_class1, X_test, feature_names=available_features, show=False)
plt.title('SHAP Feature Importance — Why Employees Leave')
plt.tight_layout()
plt.savefig('outputs/shap_importance.png', dpi=150)
plt.show()
print('SHAP analysis complete — Model is explainable!')

NameError: name 'shap' is not defined

In [ ]:
# Mean |SHAP| bar plot
plt.figure(figsize=(9, 5))
shap.plots.bar(shap_exp_class1, show=False)
plt.title('Mean |SHAP| — Global Feature Importance')
plt.tight_layout()
plt.savefig('outputs/shap_bar.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Explain a single employee prediction
sample_idx      = 0
sample_employee = X_test.iloc[sample_idx:sample_idx + 1]
prediction      = model.predict(sample_employee)[0]
probability     = model.predict_proba(sample_employee)[0]

print('Sample Employee Prediction:')
print(f"  - Risk of Leaving : {'HIGH' if prediction == 1 else 'LOW'}")
print(f'  - Probability     : {probability[1]:.1%}')

sv_sample = explainer.shap_values(sample_employee)
if isinstance(sv_sample, list):
    sv_sample_class1 = sv_sample[1][0]
elif sv_sample.ndim == 3:
    sv_sample_class1 = sv_sample[0, :, 1]
else:
    sv_sample_class1 = sv_sample[0]

print('\nKey Factors (SHAP values):')
for feat, val in sorted(
    zip(available_features, sv_sample_class1),
    key=lambda x: abs(x[1]), reverse=True
)[:5]:
    direction = 'increases' if val > 0 else 'decreases'
    print(f'  - {feat}: {val:+.3f} {direction} leaving risk')

NameError: name 'X_test' is not defined

## 7. Fairness Audit (Ethics)

In [ ]:
# Sensitive columns are used ONLY here for auditing — not model inputs.
df_audit = df.copy()
df_audit['Prediction']  = model.predict(X)
df_audit['Probability'] = model.predict_proba(X)[:, 1]

print('Fairness Audit — Prediction Rates by Sex:')
sex_rates = df_audit.groupby('Sex')['Prediction'].mean().sort_values(ascending=False)
print(sex_rates)
disparity_sex = sex_rates.max() - sex_rates.min()
flag_sex = 'review needed' if disparity_sex > 0.10 else 'within threshold'
print(f'  Disparity (max-min): {disparity_sex:.3f} ({flag_sex})')

print('\nFairness Audit — Prediction Rates by Race:')
race_rates = df_audit.groupby('RaceDesc')['Prediction'].mean().sort_values(ascending=False)
print(race_rates)
disparity_race = race_rates.max() - race_rates.min()
flag_race = 'review needed' if disparity_race > 0.20 else 'within threshold'
print(f'  Disparity (max-min): {disparity_race:.3f} ({flag_race})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sex_rates.plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'], edgecolor='black')
axes[0].set_title('Predicted Turnover Rate by Sex')
axes[0].set_ylabel('Fraction Predicted to Leave')
axes[0].set_ylim(0, 1)
axes[0].axhline(y_train.mean(), color='red', linestyle='--', label='Overall mean')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=0)

race_rates.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Predicted Turnover Rate by Race')
axes[1].set_ylabel('Fraction Predicted to Leave')
axes[1].set_ylim(0, 1)
axes[1].axhline(y_train.mean(), color='red', linestyle='--', label='Overall mean')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('outputs/fairness_audit.png', dpi=150)
plt.show()
print('Fairness audit plots saved to outputs/fairness_audit.png')

## 8. Demo — Predict Employee Risk

In [ ]:
def predict_turnover_risk(emp_data):
    """
    Predict turnover risk for a single employee.

    Parameters
    ----------
    emp_data : dict — keys must match available_features

    Returns
    -------
    risk : str   — HIGH / MEDIUM / LOW
    prob : float — probability of leaving
    """
    X_new = pd.DataFrame([emp_data]).reindex(columns=available_features, fill_value=0)
    prob  = model.predict_proba(X_new)[0][1]

    if prob > 0.60:
        risk = 'HIGH RISK'
    elif prob > 0.35:
        risk = 'MEDIUM RISK'
    else:
        risk = 'LOW RISK'

    return risk, prob


# Example: a potentially at-risk employee
sample = {
    'MarriedID'              : 0,
    'MaritalStatusID'        : 0,
    'GenderID'               : 1,
    'DeptID'                 : 6,
    'PerfScoreID'            : 2,
    'PayRate'                : 20.0,
    'PositionID'             : 19,
    'FromDiversityJobFairID' : 0,
    'ManagerID'              : 3,
    'EngagementSurvey'       : 1.5,
    'EmpSatisfaction'        : 1,
    'SpecialProjectsCount'   : 0,
    'DaysLateLast30'         : 8,
    'Tenure_Days'            : 365,
    'DaysSinceReview'        : 400,
    'HispanicLatino_Enc'     : 0,
    'CitizenDesc_Enc'        : 0,
    'State_Enc'              : 0,
    'PerformanceScore_Enc'   : 1,
    'MaritalDesc_Enc'        : 0,
}

risk, prob = predict_turnover_risk(sample)

print('=' * 52)
print('Employee Turnover Risk Prediction')
print('=' * 52)
print(f'\nRisk Level            : {risk}')
print(f'Probability of Leaving: {prob:.1%}')
print('\nRecommendations:')
if prob > 0.60:
    print('  - Urgent 1-on-1 to surface and address concerns')
    print('  - Review compensation vs. market rate')
    print('  - Consider a tailored retention package')
elif prob > 0.35:
    print('  - Schedule a quarterly check-in')
    print('  - Explore career development opportunities')
else:
    print('  - Continue regular engagement activities')
    print('  - Recognise contributions to maintain morale')

## 9. Model Card

In [ ]:
# Print live, data-driven metrics for the Model Card
acc_cv = cv_results['test_accuracy'].mean()
f1_cv  = cv_results['test_f1_macro'].mean()
auc_cv = cv_results['test_roc_auc'].mean()

print('Model Card — Live Metrics (5-Fold CV)')
print(f'  Accuracy  : {acc_cv:.1%}')
print(f'  F1-Macro  : {f1_cv:.3f}')
print(f'  ROC-AUC   : {auc_cv:.3f}')

NameError: name 'cv_results' is not defined

### Model Card: HR Turnover Prediction Model

| Property | Value |
|----------|-------|
| **Model Type** | Random Forest Classifier (class_weight=balanced) |
| **Purpose** | Predict employee turnover risk |
| **Training Data** | HR Dataset v13 (~310 employees) |
| **Accuracy (5-CV)** | See live metrics cell above |
| **Features Used** | Engagement, Satisfaction, Performance, Pay, Tenure, Lateness, Demographics (encoded) |
| **Excluded (leakage)** | EmpStatusID, EmploymentStatus, DateofTermination, TermReason |
| **Sensitive Attributes** | Sex, RaceDesc — used for fairness audit only, NOT model inputs |
| **Explainability** | SHAP TreeExplainer — global + per-prediction |
| **Fairness** | Demographic parity audit by Sex and Race |
| **Limitations** | Small dataset (310 rows); generalisation to large organisations is unverified |

### Data Card

| Property | Value |
|----------|-------|
| **Source** | Kaggle — Human Resources Data Set (Rich Huebner) |
| **Privacy** | PII removed (names, IDs, DOB, ZIP) |
| **Pseudonymisation** | Salted SHA-256 on quasi-identifiers |
| **Compliance** | GDPR Art. 25 — Data Protection by Design |
| **Sensitive Attrs** | Sex, RaceDesc — preserved for fairness audit only |

## 10. Conclusion & Next Steps

### Summary

**Data Leakage Fixed:** Removed `EmpStatusID`, `EmploymentStatus`, `DateofTermination`, `TermReason` — all directly encode the target and caused the original 100% (meaningless) accuracy.

**Cybersecurity / GDPR:** PII removed; quasi-identifiers pseudonymised with a salted SHA-256 hash (non-reversible).

**Feature Engineering:** Tenure, days-since-review, and encoded categorical columns added — giving the model genuine predictive signal.

**Robust Evaluation:** Stratified 5-fold cross-validation + hold-out test set; class imbalance handled via `class_weight='balanced'`.

**Explainable AI:** SHAP global summary + per-employee explanations using the `shap.Explanation` API (compatible with new and old SHAP versions).

**Fairness Audit:** Demographic parity measured and visualised by Sex and Race.

**Demo:** `predict_turnover_risk()` with actionable HR recommendations.

### Recommendations for Presentation

1. **Pitch:** "AI that HR can trust, understand, and legally deploy."
2. **Demo:** Run `predict_turnover_risk()` live with different inputs.
3. **Ethics:** Show the fairness audit plots and disparity scores.
4. **Innovation:** Highlight XAI (SHAP) + GDPR-safe pseudonymisation.

### Potential Next Steps

- Apply SMOTE or collect more data to address the 2:1 class imbalance.
- Try XGBoost / LightGBM and compare ROC-AUC.
- Integrate IBM AI Fairness 360 for production-grade fairness assessment.
- Tune the classification threshold to optimise recall for the at-risk class.